In [0]:
use catalog identifier(:catalog)

In [0]:
create or replace temp view dq_bronze_checks AS
WITH expected AS (
  SELECT * FROM VALUES ('region'),('nation'),('airline'),('route'),
                       ('airline_route'),('passenger'),('booking'),('flight_booking') AS v(t)
),
bronze_counts AS (
  SELECT 'region' AS t, count(*) AS cnt FROM bronze.region
  UNION ALL SELECT 'nation', count(*) FROM bronze.nation
  UNION ALL SELECT 'airline', count(*) FROM bronze.airline
  UNION ALL SELECT 'route', count(*) FROM bronze.route
  UNION ALL SELECT 'airline_route', count(*) FROM bronze.airline_route
  UNION ALL SELECT 'passenger', count(*) FROM bronze.passenger
  UNION ALL SELECT 'booking', count(*) FROM bronze.booking
  UNION ALL SELECT 'flight_booking', count(*) FROM bronze.flight_booking
),
existing AS (
  SELECT lower(table_name) AS table_name
  FROM information_schema.tables
  WHERE table_schema = 'bronze'
)
SELECT 'B01' AS check_id, 'Bronze completeness' AS category,
       'table_exists_in_bronze' AS check_name, e.t AS object_name,
       CASE WHEN x.table_name IS NULL THEN 1 ELSE 0 END AS fail_count,
       'CRITICAL' AS severity
FROM expected e
LEFT JOIN existing x ON x.table_name = e.t
UNION ALL
SELECT 'B02', 'Bronze completeness', 'bronze_table_not_empty', e.t,
       CASE WHEN c.cnt IS NULL OR c.cnt = 0 THEN 1 ELSE 0 END,
       'CRITICAL'
FROM expected e
LEFT JOIN bronze_counts c ON c.t = e.t

In [0]:
create or replace temp view dq_pk_checks AS
SELECT 'S01' AS check_id, 'Primary keys' AS category, 'pk_not_null' AS check_name,
       'silver.region(regionkey)' AS object_name,
       count(*) - count(regionkey) AS fail_count, 'CRITICAL' AS severity
FROM silver.region
UNION ALL SELECT 'S02', 'Primary keys', 'pk_unique', 'silver.region(regionkey)',
       count(regionkey) - count(DISTINCT regionkey), 'CRITICAL' FROM silver.region
UNION ALL SELECT 'S03', 'Primary keys', 'pk_not_null', 'silver.nation(nationkey)',
       count(*) - count(nationkey), 'CRITICAL' FROM silver.nation
UNION ALL SELECT 'S04', 'Primary keys', 'pk_unique', 'silver.nation(nationkey)',
       count(nationkey) - count(DISTINCT nationkey), 'CRITICAL' FROM silver.nation
UNION ALL SELECT 'S05', 'Primary keys', 'pk_not_null', 'silver.airline(airlineid)',
       count(*) - count(airlineid), 'CRITICAL' FROM silver.airline
UNION ALL SELECT 'S06', 'Primary keys', 'pk_unique', 'silver.airline(airlineid)',
       count(airlineid) - count(DISTINCT airlineid), 'CRITICAL' FROM silver.airline
UNION ALL SELECT 'S07', 'Primary keys', 'pk_not_null', 'silver.route(routeid)',
       count(*) - count(routeid), 'CRITICAL' FROM silver.route
UNION ALL SELECT 'S08', 'Primary keys', 'pk_unique', 'silver.route(routeid)',
       count(routeid) - count(DISTINCT routeid), 'CRITICAL' FROM silver.route
UNION ALL SELECT 'S09', 'Primary keys', 'pk_not_null', 'silver.passenger(passengerid)',
       count(*) - count(passengerid), 'CRITICAL' FROM silver.passenger
UNION ALL SELECT 'S10', 'Primary keys', 'pk_unique', 'silver.passenger(passengerid)',
       count(passengerid) - count(DISTINCT passengerid), 'CRITICAL' FROM silver.passenger
UNION ALL SELECT 'S11', 'Primary keys', 'pk_not_null', 'silver.booking(bookingid)',
       count(*) - count(bookingid), 'CRITICAL' FROM silver.booking
UNION ALL SELECT 'S12', 'Primary keys', 'pk_unique', 'silver.booking(bookingid)',
       count(bookingid) - count(DISTINCT bookingid), 'CRITICAL' FROM silver.booking
-- composite PKs: rows with null key parts + duplicate keys counted over non-null rows only
UNION ALL SELECT 'S13', 'Primary keys', 'pk_not_null', 'silver.airline_route(airlineid,routeid)',
       count(*) FILTER (WHERE airlineid IS NULL OR routeid IS NULL), 'CRITICAL' FROM silver.airline_route
UNION ALL SELECT 'S14', 'Primary keys', 'pk_unique', 'silver.airline_route(airlineid,routeid)',
       count(*) FILTER (WHERE airlineid IS NOT NULL AND routeid IS NOT NULL)
         - count(DISTINCT CASE WHEN airlineid IS NOT NULL AND routeid IS NOT NULL
                               THEN concat(airlineid, '|', routeid) END), 'CRITICAL'
FROM silver.airline_route
UNION ALL SELECT 'S15', 'Primary keys', 'pk_not_null', 'silver.flight_booking(bookingid,linenumber)',
       count(*) FILTER (WHERE bookingid IS NULL OR linenumber IS NULL), 'CRITICAL' FROM silver.flight_booking
UNION ALL SELECT 'S16', 'Primary keys', 'pk_unique', 'silver.flight_booking(bookingid,linenumber)',
       count(*) FILTER (WHERE bookingid IS NOT NULL AND linenumber IS NOT NULL)
         - count(DISTINCT CASE WHEN bookingid IS NOT NULL AND linenumber IS NOT NULL
                               THEN concat(bookingid, '|', linenumber) END), 'CRITICAL'
FROM silver.flight_booking

In [0]:

create or replace temp view dq_fact_checks AS
SELECT 'F01' AS check_id, 'Fact table' AS category, 'surrogate_key_not_null' AS check_name,
       'gold.fact_bookings.booking_id' AS object_name,
       count(*) FILTER (WHERE booking_id IS NULL) AS fail_count, 'CRITICAL' AS severity
FROM gold.fact_bookings
UNION ALL SELECT 'F02', 'Fact table', 'surrogate_key_not_null', 'gold.fact_bookings.line_number',
       count(*) FILTER (WHERE line_number IS NULL), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F03', 'Fact table', 'surrogate_key_not_null', 'gold.fact_bookings.passenger_id',
       count(*) FILTER (WHERE passenger_id IS NULL), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F04', 'Fact table', 'surrogate_key_not_null', 'gold.fact_bookings.route_id',
       count(*) FILTER (WHERE route_id IS NULL), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F05', 'Fact table', 'surrogate_key_not_null', 'gold.fact_bookings.airline_id',
       count(*) FILTER (WHERE airline_id IS NULL), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F06', 'Fact table', 'surrogate_key_not_null', 'gold.fact_bookings.passenger_nation_key',
       count(*) FILTER (WHERE passenger_nation_key IS NULL), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F07', 'Fact table', 'surrogate_key_not_null', 'gold.fact_bookings.airline_nation_key',
       count(*) FILTER (WHERE airline_nation_key IS NULL), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F08', 'Business rules', 'ticket_price_gt_0', 'gold.fact_bookings.ticket_price',
       count(*) FILTER (WHERE ticket_price IS NULL OR ticket_price <= 0), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F09', 'Business rules', 'discount_between_0_and_1', 'gold.fact_bookings.discount_rate',
       count(*) FILTER (WHERE discount_rate IS NULL OR discount_rate < 0 OR discount_rate > 1), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F10', 'Business rules', 'seats_ge_1', 'gold.fact_bookings.seat_count',
       count(*) FILTER (WHERE seat_count IS NULL OR seat_count < 1), 'CRITICAL' FROM gold.fact_bookings
UNION ALL SELECT 'F11', 'Business rules', 'net_revenue_not_null', 'gold.fact_bookings.net_revenue',
       count(*) FILTER (WHERE net_revenue IS NULL), 'CRITICAL' FROM gold.fact_bookings

In [0]:
-- CHECK 4: Gold dimensions non-empty and dimension keys unique
create or replace temp view dq_dim_checks AS
SELECT 'D01' AS check_id, 'Dimensions' AS category, 'dim_not_empty' AS check_name,
       'gold.dim_geography' AS object_name,
       CASE WHEN count(*) = 0 THEN 1 ELSE 0 END AS fail_count, 'CRITICAL' AS severity
FROM gold.dim_geography
UNION ALL SELECT 'D02', 'Dimensions', 'dim_key_unique', 'gold.dim_geography(nation_key)',
       count(*) - count(DISTINCT nation_key), 'CRITICAL' FROM gold.dim_geography
UNION ALL SELECT 'D03', 'Dimensions', 'dim_not_empty', 'gold.dim_passenger',
       CASE WHEN count(*) = 0 THEN 1 ELSE 0 END, 'CRITICAL' FROM gold.dim_passenger
UNION ALL SELECT 'D04', 'Dimensions', 'dim_key_unique', 'gold.dim_passenger(passenger_id)',
       count(*) - count(DISTINCT passenger_id), 'CRITICAL' FROM gold.dim_passenger
UNION ALL SELECT 'D05', 'Dimensions', 'dim_not_empty', 'gold.dim_route',
       CASE WHEN count(*) = 0 THEN 1 ELSE 0 END, 'CRITICAL' FROM gold.dim_route
UNION ALL SELECT 'D06', 'Dimensions', 'dim_key_unique', 'gold.dim_route(route_id)',
       count(*) - count(DISTINCT route_id), 'CRITICAL' FROM gold.dim_route
UNION ALL SELECT 'D07', 'Dimensions', 'dim_not_empty', 'gold.dim_airline',
       CASE WHEN count(*) = 0 THEN 1 ELSE 0 END, 'CRITICAL' FROM gold.dim_airline
UNION ALL SELECT 'D08', 'Dimensions', 'dim_key_unique', 'gold.dim_airline(airline_id)',
       count(*) - count(DISTINCT airline_id), 'CRITICAL' FROM gold.dim_airline

In [0]:
-- CHECK 5: Gold revenue and row counts reconcile with Silver (relative tolerance <= 0.01%)
create or replace temp view dq_recon_checks AS
WITH recon AS (
  SELECT 'fact_row_count' AS metric,
         (SELECT count(*) FROM gold.fact_bookings) AS gold_value,
         (SELECT count(*) FROM silver.flight_booking) AS silver_value
  UNION ALL
  SELECT 'net_revenue_total',
         (SELECT round(sum(net_revenue), 2) FROM gold.fact_bookings),
         (SELECT round(sum(ticketprice * (1 - discount)), 2) FROM silver.flight_booking)
  UNION ALL
  SELECT 'gross_revenue_total',
         (SELECT round(sum(gross_revenue), 2) FROM gold.fact_bookings),
         (SELECT round(sum(ticketprice), 2) FROM silver.flight_booking)
  UNION ALL
  SELECT 'distinct_bookings',
         (SELECT count(DISTINCT booking_id) FROM gold.fact_bookings),
         (SELECT count(DISTINCT bookingid) FROM silver.flight_booking)
  UNION ALL
  SELECT 'active_passengers',
         (SELECT count(DISTINCT passenger_id) FROM gold.fact_bookings),
         (SELECT count(DISTINCT bk.passengerid)
          FROM silver.flight_booking fb JOIN silver.booking bk ON bk.bookingid = fb.bookingid)
)
SELECT concat('R', lpad(cast(row_number() OVER (ORDER BY metric) AS STRING), 2, '0')) AS check_id,
       'Gold-Silver reconciliation' AS category,
       'recon_rel_diff_le_0.01pct' AS check_name,
       metric AS object_name, gold_value, silver_value,
       round(abs(gold_value - silver_value) / greatest(abs(silver_value), 1) * 100, 6) AS rel_diff_pct,
       CASE WHEN abs(gold_value - silver_value) <= 0.0001 * greatest(abs(silver_value), 1)
            THEN 0 ELSE 1 END AS fail_count,
       'CRITICAL' AS severity
FROM recon

In [0]:
-- CHECK 6: Rejected-record counts per table (report only — NOT a failure condition)
create or replace temp view dq_quarantine_report AS
SELECT 'Q01' AS check_id, 'Rejected records' AS category, 'quarantined_rows' AS check_name,
       'silver.region_quarantine' AS object_name,
       (SELECT count(*) FROM silver.region_quarantine) AS fail_count, 'REPORT' AS severity
UNION ALL SELECT 'Q02', 'Rejected records', 'quarantined_rows', 'silver.nation_quarantine',
       (SELECT count(*) FROM silver.nation_quarantine), 'REPORT'
UNION ALL SELECT 'Q03', 'Rejected records', 'quarantined_rows', 'silver.airline_quarantine',
       (SELECT count(*) FROM silver.airline_quarantine), 'REPORT'
UNION ALL SELECT 'Q04', 'Rejected records', 'quarantined_rows', 'silver.route_quarantine',
       (SELECT count(*) FROM silver.route_quarantine), 'REPORT'
UNION ALL SELECT 'Q05', 'Rejected records', 'quarantined_rows', 'silver.airline_route_quarantine',
       (SELECT count(*) FROM silver.airline_route_quarantine), 'REPORT'
UNION ALL SELECT 'Q06', 'Rejected records', 'quarantined_rows', 'silver.passenger_quarantine',
       (SELECT count(*) FROM silver.passenger_quarantine), 'REPORT'
UNION ALL SELECT 'Q07', 'Rejected records', 'quarantined_rows', 'silver.booking_quarantine',
       (SELECT count(*) FROM silver.booking_quarantine), 'REPORT'
UNION ALL SELECT 'Q08', 'Rejected records', 'quarantined_rows', 'silver.flight_booking_quarantine',
       (SELECT count(*) FROM silver.flight_booking_quarantine), 'REPORT'

In [0]:
-- Persist all check results to gold.data_quality_results
create or replace table gold.data_quality_results AS
SELECT check_id, category, check_name, object_name, fail_count, severity,
       CASE WHEN severity = 'REPORT' THEN 'REPORTED'
            WHEN fail_count = 0 THEN 'PASS' ELSE 'FAIL' END AS status,
       current_timestamp() AS checked_at
FROM (
  SELECT check_id, category, check_name, object_name, fail_count, severity FROM dq_bronze_checks
  UNION ALL SELECT check_id, category, check_name, object_name, fail_count, severity FROM dq_pk_checks
  UNION ALL SELECT check_id, category, check_name, object_name, fail_count, severity FROM dq_fact_checks
  UNION ALL SELECT check_id, category, check_name, object_name, fail_count, severity FROM dq_dim_checks
  UNION ALL SELECT check_id, category, check_name, object_name, fail_count, severity FROM dq_recon_checks
  UNION ALL SELECT check_id, category, check_name, object_name, fail_count, severity FROM dq_quarantine_report
)

num_affected_rows,num_inserted_rows


In [0]:
-- PIPELINE-STOPPING GATE: raises an error when any critical check fails
SELECT assert_true(
         (SELECT count(*) FROM gold.data_quality_results
          WHERE severity = 'CRITICAL' AND status = 'FAIL') = 0,
         (SELECT concat('DATA QUALITY GATE FAILED: ', count(*), ' critical check(s) failed. Details: ',
                        string_agg(concat(object_name, ' [', check_name, '] fail_count=', cast(fail_count AS STRING)), '; '))
          FROM gold.data_quality_results
          WHERE severity = 'CRITICAL' AND status = 'FAIL')
       ) AS dq_gate

dq_gate
null
